<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Machine-Learning/18-causal-machine-learning.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回机器学习总览](Machine Learning.html)


## **因果机器学习**

预测机器学习通过学习稳定关联来预测结果。**因果机器学习（causal machine learning）**提出的是另一个问题：如果主动改变一项治疗、政策、价格、信息或系统组件，结果将如何变化？灵活的预测模型依然有用，但它被嵌入因果设计之中，而不能单独作为因果关系的证据。

因果分析应从四个对象开始：

1. 一个定义清楚的**处理或干预（treatment or intervention）** $T$；
2. 在指定时间范围内测量的**结果（outcome）** $Y$；
3. 结论所适用的**目标总体（target population）**；
4. 一个**因果估计目标（causal estimand）**，例如平均处理效应或条件处理效应。

只有确定这些对象之后，才应该选择估计器。“使用 causal forest”并不是一个因果问题，正如“使用神经网络”并不是一个完整的预测问题定义。

严谨的因果工作流会区分五个在应用中经常被混为一谈的层次：

| 层次 | 核心问题 | 典型输出 |
|---|---|---|
| 问题设定 | 目标干预、总体、结果时间范围和效应究竟是什么？ | 目标试验与估计目标 |
| 因果模型或研究设计 | 哪些变量导致处理与结果，或哪种分配机制产生可比较组？ | DAG、随机设计或准实验设计 |
| 识别 | 能否用观测数据分布表示因果估计目标？ | 调整公式、工具变量估计目标、RD 对比，或无法识别的证明 |
| 估计 | 如何用有限样本近似已经识别的量？ | 回归、加权、匹配、正交分数或森林估计 |
| 推断与压力测试 | 估计有多不确定，对假设违反有多敏感？ | 标准误、置信区间、诊断、安慰剂检验与敏感性分析 |

这种区分并非文字游戏。识别是特定假设下的逻辑结论，估计则是统计近似。即使一个低方差估计器估计了错误的识别量，它也只是在非常精确地犯错。

### **预测、关联与因果**

同一组变量可以支持几种本质不同的问题：

| 问题类型 | 数学对象 | 示例 | 如何验证？ |
|---|---|---|---|
| 预测 | $P(Y\mid X)$ 或 $\mathbb E[Y\mid X]$ | 谁可能流失？ | 部署分布下的样本外预测性能 |
| 关联 | 观测群体之间的对比 | 接受联系与未接受联系的用户，流失率有何差异？ | 是否准确描述观测数据 |
| 干预 | $P(Y\mid do(T=t))$ 或 $\mathbb E[Y(t)]$ | 如果所有用户都接受联系策略 $t$，流失率会是多少？ | 研究设计与识别假设 |
| 反事实 | $Y_i(t)$ 与另一个未实现处理结果的对比 | 如果这位用户没有接受实际收到的联系，他是否还会留下？ | 具有更强假设的结构模型或潜在结果模型 |

<div class="diagram-scroll">

![预测、干预与反事实问题代表不同层次的因果推理。](assets/causal-question-ladder.svg){fig-alt="预测估计可能发生的结果，干预询问主动设置处理后会发生什么变化，反事实推理则比较同一个体已经实现与未实现的结果。"}

</div>

只有在恰当的研究设计或调整论证下，关联才等于因果效应。如果高风险用户更容易接受干预，即使干预有帮助，处理组的结果仍可能更差。反过来，看似有益的关联也可能完全由有利的选择机制产生。

<details>
<summary><strong>Python：展示混杂如何使观测关联偏离处理效应</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
n = 20_000

# Engagement affects both campaign assignment and future spending.
engagement = rng.normal(size=n)
propensity = 1 / (1 + np.exp(-(-0.2 + 1.4 * engagement)))
treatment = rng.binomial(1, propensity)

true_effect = 2.0
outcome = (
    5.0
    + true_effect * treatment
    + 3.0 * engagement
    + rng.normal(scale=1.0, size=n)
)

# This is only an observed-group contrast.
naive_difference = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()

# Under the simulated no-hidden-confounding assumption, adjusting for engagement
# recovers the treatment coefficient.
adjusted_model = LinearRegression().fit(
    np.column_stack([treatment, engagement]),
    outcome,
)
adjusted_effect = adjusted_model.coef_[0]

print(f"True causal effect:        {true_effect:.3f}")
print(f"Naive treated-control gap: {naive_difference:.3f}")
print(f"Adjusted treatment effect: {adjusted_effect:.3f}")
```

</details>

这里调整后的系数具有因果含义，只是因为模拟过程把 engagement 设为全部共同原因，并采用了正确设定的加性结果模型。更高的交叉验证 $R^2$ 并不能证明这些假设成立。


### **结构因果模型**

**结构因果模型（structural causal model, SCM）**把每个内生变量表示为其直接原因与外生扰动的确定性函数：

$$
X_j=f_j(\operatorname{pa}_j,U_j).
$$

集合

$$
\mathcal M=(U,V,F,P_U)
$$

包含外生变量 $U$、内生变量 $V$、结构机制 $F$ 以及外生扰动的分布 $P_U$。在无环 SCM 中，按照拓扑顺序计算这些方程即可生成观测分布。

关键的因果假设是**模块性（modularity）**：每个结构方程描述一个相对独立的机制，至少在概念上可以改变其中一个机制，而不必重写其他所有机制。正是这一假设使图中的箭头具有超越统计依赖的因果含义。

#### **因果 DAG 与结构方程**

有向无环图（directed acyclic graph, DAG）概括了哪些变量是结构方程中的直接原因。例如，

$$
C=f_C(U_C),\qquad
T=f_T(C,U_T),\qquad
Y=f_Y(T,C,U_Y)
$$

对应 $C\rightarrow T$、$C\rightarrow Y$ 和 $T\rightarrow Y$。DAG 省略具体函数形式，但显露出与因果识别有关的路径。

干预 $do(T=t)$ 通过下式替换自然的处理方程，从而产生一个修改后的模型：

$$
T:=t.
$$

所有进入 $T$ 的箭头都被移除，而 $C$ 与 $Y$ 的方程保持不变。

<div class="diagram-scroll">

![外科手术式干预替换处理方程，同时保留其他机制。](assets/scm-surgical-intervention.svg){fig-alt="通过用固定处理值替换自然处理机制，观测 SCM 被转换为一个干预世界。"}

</div>

<details>
<summary><strong>Python：比较观测斜率与外科手术式干预</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(7)
n = 50_000

# U is a common cause of treatment and outcome.
u = rng.normal(size=n)
treatment_noise = rng.normal(size=n)
outcome_noise = rng.normal(size=n)

treatment = 1.2 * u + treatment_noise
outcome = 3.0 * treatment + 2.0 * u + outcome_noise

observational_slope = LinearRegression().fit(
    treatment.reshape(-1, 1), outcome
).coef_[0]
adjusted_slope = LinearRegression().fit(
    np.column_stack([treatment, u]), outcome
).coef_[0]

# do(T=t) replaces the treatment equation but leaves U and the outcome
# mechanism unchanged. Using common noise makes the contrast easy to audit.
y_do_0 = 3.0 * 0.0 + 2.0 * u + outcome_noise
y_do_1 = 3.0 * 1.0 + 2.0 * u + outcome_noise
interventional_effect = np.mean(y_do_1 - y_do_0)

print(f"Observational slope:       {observational_slope:.3f}")
print(f"Adjusted slope:            {adjusted_slope:.3f}")
print(f"E[Y|do(T=1)]-E[Y|do(T=0)]: {interventional_effect:.3f}")
```

</details>

不能在观察相关性后画几条箭头，就声称学到了 DAG。箭头方向编码了时间、科学、工程或研究设计知识。两位分析者可以拟合同一个观测分布，却因为对未观测机制持有不同假设而得出不同的因果结论。

#### **混杂变量、媒介变量与碰撞变量**

变量的角色由所研究的路径决定，而不是由数据表中的列名决定：

- **混杂变量（confounder）**是后门路径上的共同原因，例如 $T\leftarrow C\rightarrow Y$。对充分的处理前混杂变量集合进行调整，可以阻断非因果关联。
- **媒介变量（mediator）**位于因果路径上，例如 $T\rightarrow M\rightarrow Y$。对 $M$ 进行调整会移除经由它传递的效应，因此不能估计总效应。
- **碰撞变量（collider）**有两条箭头指向它，例如 $T\rightarrow S\leftarrow Y$。这条路径天然关闭，但以 $S$ 为条件可能打开一条人为关联。
- **代理变量（proxy）**可能只是不完全地测量混杂变量，无法将混杂彻底控制。
- **工具变量（instrument）**影响处理；在强排除限制与独立性条件下，它只能通过处理影响结果。

<div class="diagram-scroll">

![混杂变量、媒介变量与碰撞变量要求不同的调整决策。](assets/dag-variable-roles.svg){fig-alt="混杂变量同时导致处理与结果，媒介变量传递部分处理效应，碰撞变量则同时由处理与结果导致。"}

</div>

这些角色可以用 **d-separation** 准确定义。如果路径满足以下任一条件，它就被调整集合 $Z$ 阻断：

1. 路径包含一个属于 $Z$ 的非碰撞节点；
2. 路径包含一个碰撞节点，而且该碰撞节点及其任何后代都不属于 $Z$。

如果两项都不满足，该路径在给定 $Z$ 后仍然开放，也称为 **d-connected**。以非碰撞节点为条件会阻断路径；以碰撞节点或其后代为条件反而可能打开路径。正因如此，依据相关性、特征重要性或预测增益选择调整变量，并不是有效的因果调整策略。

对于 $T$ 对 $Y$ 的总效应，**正规后门路径（proper backdoor path）**以一条进入 $T$ 的箭头开始。充分调整集合必须阻断所有这类路径，并排除 $T$ 的后代。一个问题可能存在多个有效调整集合。**最小充分调整集合**是指删除其中任何变量都会重新打开后门路径的集合；它通常更值得优先采用，因为不必要的调整可能增加方差、放大测量误差或恶化重叠性。

| 变量相对于 $T\rightarrow Y$ 问题的角色 | 估计总效应时是否调整？ | 原因 |
|---|---|---|
| $T$ 与 $Y$ 的处理前共同原因 | 通常需要 | 阻断非因果后门路径 |
| 与 $T$ 无关但导致 $Y$ 的变量 | 可选 | 可以改善精度而不改变识别 |
| 与 $Y$ 无关但强烈导致 $T$ 的变量 | 通常不必要 | 可能增加权重方差，并放大残余隐藏偏差 |
| 由 $T$ 导致的媒介变量 | 不调整 | 会移除总因果路径的一部分 |
| 碰撞变量或碰撞变量的后代 | 不调整 | 可能产生选择诱导的关联 |
| 在结果之前测量、但属于处理后变量 | 估计总效应时不调整 | 即使时间戳早于 $Y$，它仍然是处理后的结果 |

因此，“控制所有可用特征”并不安全。处理发生后记录的特征可能是媒介变量、碰撞变量、选择的结果，或来自结果的泄漏。应在拟合模型之前，根据因果问题与因果图论证调整变量集合。


### **干预与 Do-Operator**

条件分布

$$
P(Y\mid T=t)
$$

描述自然处理值等于 $t$ 的个体。干预分布

$$
P(Y\mid do(T=t))
$$

则描述一个从外部把处理设置为 $t$ 的修改后系统。两者通常不同，因为条件化并不会移除处理本身的原因。

#### **观测分布与干预分布**

如果一个 DAG 可以分解为

$$
P(v_1,\ldots,v_p)=\prod_{j=1}^{p}P(v_j\mid \operatorname{pa}_j),
$$

那么对 $T$ 进行干预会产生一个**截断分解（truncated factorization）**：自然处理机制对应的因子被移除，$T$ 被固定，其他机制全部保留。

<div class="diagram-scroll">

![条件化选择自然接受处理的个体，而干预会改变处理分配机制。](assets/conditioning-versus-intervention.svg){fig-alt="条件化仍然保留处理的自然原因，干预则切断所有进入处理变量的箭头，并强制指定其取值。"}

</div>

**识别（identification）**询问的是：在明确假设下，能否把干预目标完全改写为观测数据分布中的量。**估计（estimation）**随后才使用有限数据近似这个已经识别的表达式。这个顺序非常重要：

> 无法识别的因果效应，不能被更准确的预测模型拯救。

#### **后门调整与前门调整**

如果处理前变量集合 $C$ 阻断了从 $T$ 到 $Y$ 的全部后门路径，并且不包含 $T$ 的后代，那么

$$
P(Y\mid do(T=t))
=
\sum_c P(Y\mid T=t,C=c)P(C=c).
$$

当 $C$ 连续或高维时，求和会变为积分，可以通过结果回归、加权、匹配、分层或双重稳健方法估计。

<div class="diagram-scroll">

![后门调整使用观测到的共同原因，前门识别则使用有效的媒介变量。](assets/backdoor-frontdoor-identification.svg){fig-alt="后门图调整处理与结果的共同原因；前门图则在处理与结果存在隐藏混杂时，使用完全传递效应的媒介变量完成识别。"}

</div>

<details>
<summary><strong>Python：通过结果标准化估计经过后门调整的 ATE</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(12)
n = 12_000
x = rng.normal(size=(n, 3))

# Treatment selection depends nonlinearly on observed pre-treatment covariates.
logit = 0.2 + 0.9 * x[:, 0] - 0.7 * x[:, 1] + 0.4 * x[:, 0] * x[:, 1]
propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, propensity)

true_cate = 1.5 + 0.8 * np.tanh(x[:, 0])
baseline = 2.0 + x[:, 0] ** 2 - 0.8 * x[:, 1] + 0.5 * x[:, 2]
outcome = baseline + true_cate * treatment + rng.normal(scale=1.0, size=n)

# Learn E[Y | T, X], then standardize both treatment states over the same X.
outcome_model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=20,
    random_state=12,
    n_jobs=-1,
).fit(np.column_stack([treatment, x]), outcome)

mu_1 = outcome_model.predict(np.column_stack([np.ones(n), x]))
mu_0 = outcome_model.predict(np.column_stack([np.zeros(n), x]))

naive = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()
g_formula_ate = np.mean(mu_1 - mu_0)

print(f"True sample ATE:       {true_cate.mean():.3f}")
print(f"Naive observed gap:    {naive:.3f}")
print(f"Standardized RF ATE:   {g_formula_ate:.3f}")
```

</details>

即使存在未观测的 $T$-$Y$ 混杂，只要媒介变量 $M$ 满足一组严格条件，**前门准则（frontdoor criterion）**仍可能识别因果效应：$T$ 在没有未阻断混杂的情况下导致 $M$；$M$ 截断了从 $T$ 到 $Y$ 的全部有向路径；并且从 $M$ 到 $Y$ 的全部后门路径都能被 $T$ 阻断。此时

$$
P(y\mid do(t))
=
\sum_m P(m\mid t)
\sum_{t'}P(y\mid m,t')P(t').
$$

<details>
<summary><strong>Python：在模拟系统中恢复前门效应</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression

rng = np.random.default_rng(21)
n = 60_000

# U confounds treatment and outcome but does not directly affect the mediator.
u = rng.normal(size=n)
p_t = 1 / (1 + np.exp(-(1.2 * u)))
treatment = rng.binomial(1, p_t)

p_m = 1 / (1 + np.exp(-(-0.6 + 1.8 * treatment)))
mediator = rng.binomial(1, p_m)
outcome = 2.5 * mediator + 1.5 * u + rng.normal(size=n)

mediator_model = LogisticRegression().fit(treatment.reshape(-1, 1), mediator)
outcome_model = LinearRegression().fit(
    np.column_stack([mediator, treatment]), outcome
)

treatment_probability = treatment.mean()

def frontdoor_mean(t_value):
    # First average over M under the intervention T=t.
    p_m1 = mediator_model.predict_proba([[t_value]])[0, 1]
    total = 0.0
    for m_value, p_m_given_t in [(0, 1 - p_m1), (1, p_m1)]:
        # Then adjust the M->Y relation over the natural distribution of T.
        y_m_t0 = outcome_model.predict([[m_value, 0]])[0]
        y_m_t1 = outcome_model.predict([[m_value, 1]])[0]
        adjusted_y_m = (
            (1 - treatment_probability) * y_m_t0
            + treatment_probability * y_m_t1
        )
        total += p_m_given_t * adjusted_y_m
    return total

frontdoor_effect = frontdoor_mean(1) - frontdoor_mean(0)
true_effect = 2.5 * (
    1 / (1 + np.exp(-1.2)) - 1 / (1 + np.exp(0.6))
)
naive = outcome[treatment == 1].mean() - outcome[treatment == 0].mean()

print(f"True total effect:  {true_effect:.3f}")
print(f"Naive group gap:    {naive:.3f}")
print(f"Frontdoor estimate: {frontdoor_effect:.3f}")
```

</details>

前门调整并不是隐藏混杂的通用修复方法。它的假设通常比普通后门调整更难辩护，尤其是“不存在直接的处理—结果路径”和“不存在隐藏的处理—媒介混杂”这两项要求。


### **潜在结果**

潜在结果框架为每个个体 $i$ 定义

$$
Y_i(1)
\quad\text{and}\quad
Y_i(0),
$$

分别表示处理和对照条件下会出现的结果。对于二元处理，观测结果满足**一致性（consistency）**：

$$
Y_i
=
T_iY_i(1)+(1-T_i)Y_i(0).
$$

这种表示法不依赖某个具体估计器即可定义因果效应，同时也明确展现了因果推断中的缺失数据结构。

#### **反事实与因果推断的根本问题**

对单个个体而言，个体处理效应为

$$
\tau_i=Y_i(1)-Y_i(0).
$$

我们只能观察其中一个潜在结果，因此 $\tau_i$ 永远无法被直接观察。这就是**因果推断的根本问题（fundamental problem of causal inference）**。重复测量通常也不能解决它，因为时间、残留效应、学习和变化的上下文会形成不同的个体状态或处理方案。

<div class="diagram-scroll">

![处理分配揭示一个潜在结果，并使另一个结果成为反事实。](assets/potential-outcomes-missingness.svg){fig-alt="每个个体都有接受处理和未接受处理时的潜在结果，但处理分配只揭示其中一个，因此个体处理效应处于缺失状态。"}

</div>

常见缩写 SUTVA 组合了以下要求：

- **一致性**：接受实际处理后观察到的结果等于相应的潜在结果；
- **不存在隐藏的处理版本**：处理 $t$ 代表一个定义充分明确的干预；
- **不存在干扰（no interference）**：一个个体的结果不依赖其他个体的处理分配。

网络效应、共享资源、拍卖、传染病传播和推荐系统经常违反无干扰假设。此时处理定义必须包含相关的暴露映射，或者把分析单位提升到集群或网络层级。

#### **识别假设**

潜在结果定义了估计目标，却不会自动识别它。对于观测数据中的二元处理，标准调整论证需要同时满足：

1. **一致性**：若 $T=t$，则 $Y=Y(t)$；
2. **条件可交换性**：对 $t\in\{0,1\}$，有 $Y(t)\perp T\mid X$；
3. **正值性**：目标总体中的每个协变量模式都具有非零概率接受每一种处理；
4. **定义正确的抽样与测量**：$X$、$T$ 和 $Y$ 分别表示预期总体、处理版本与结果时间范围。

在这些条件下，平均潜在结果可由 g-formula 识别：

$$
\begin{aligned}
\mathbb E[Y(t)]
&=\mathbb E_X\!\left[\mathbb E[Y(t)\mid X]\right] \\
&=\mathbb E_X\!\left[\mathbb E[Y(t)\mid T=t,X]\right] \\
&=\mathbb E_X\!\left[\mathbb E[Y\mid T=t,X]\right].
\end{aligned}
$$

第一行使用迭代期望，第二行使用可交换性，第三行使用一致性。正值性保证最后一个条件期望在目标协变量分布的两种处理状态下都能从数据中学习。

这个推导也揭示了不同失败模式。如果可交换性失败，观测条件均值就不等于缺失的反事实均值；如果正值性失败，公式可能对一个更大的假想总体在代数上成立，但数据没有足够支持，估计必须依赖外推；如果处理存在隐藏版本，$Y(t)$ 本身就不是单一、连贯的潜在结果。

#### **ATE、ATT 与异质处理效应**

总体层面的因果量包括

$$
\operatorname{ATE}
=
\mathbb E[Y(1)-Y(0)],
$$

$$
\operatorname{ATT}
=
\mathbb E[Y(1)-Y(0)\mid T=1],
$$

以及条件平均处理效应

$$
\tau(x)
=
\mathbb E[Y(1)-Y(0)\mid X=x].
$$

只要处理效应具有异质性，而且处理选择改变了不同组的个体构成，ATE、ATT 与 ATC 就会不同。

<div class="diagram-scroll">

![ATE、ATT、ATC 与 CATE 在不同总体上平均处理效应。](assets/causal-estimands-populations.svg){fig-alt="ATE 面向完整总体，ATT 面向实际接受处理的个体，ATC 面向未接受处理的个体，CATE 面向具有指定协变量取值的个体。"}

</div>

<details>
<summary><strong>Python：展示 ATE、ATT 与观测组间差异为何会发生分离</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(30)
n = 100_000
x = rng.normal(size=n)

y0 = 2.0 + x + rng.normal(scale=0.8, size=n)
individual_effect = 1.0 + 1.2 * x
y1 = y0 + individual_effect

# Units with larger effects are more likely to receive treatment.
propensity = 1 / (1 + np.exp(-(-0.3 + 1.2 * x)))
treatment = rng.binomial(1, propensity)
observed_y = treatment * y1 + (1 - treatment) * y0

ate = individual_effect.mean()
att = individual_effect[treatment == 1].mean()
atc = individual_effect[treatment == 0].mean()
observed_gap = (
    observed_y[treatment == 1].mean()
    - observed_y[treatment == 0].mean()
)

print(f"ATE:                    {ate:.3f}")
print(f"ATT:                    {att:.3f}")
print(f"ATC:                    {atc:.3f}")
print(f"Observed treated gap:   {observed_gap:.3f}")
```

</details>

估计的 CATE 是条件均值，并不证明我们知道某个具体个体的处理效应。异质性分析还会引入多重检验、正则化、重叠性和策略评估问题。在同一份数据上发现并评估的子群体，很容易因为构造过程本身而显得响应强烈。


### **随机试验与观测研究**

随机化与观测数据调整不是可以互换的数据清理方法，而是不同的因果识别策略。

<div class="diagram-scroll">

![随机试验、观测研究与目标试验模拟对假设提出不同要求。](assets/randomized-observational-target-trial.svg){fig-alt="随机分配通过设计识别因果效应，观测分配依赖额外假设，目标试验模拟则明确资格、时间零点、处理与结果。"}

</div>

#### **随机化与识别**

在理想随机试验中，

$$
T\perp (Y(1),Y(0),X),
$$

因此样本均值差可以识别 ATE：

$$
\widehat{\tau}_{\mathrm{DM}}
=
\overline Y_{T=1}-\overline Y_{T=0}.
$$

协变量调整可以提高精度，但随机化本身才是识别的来源。实践中的复杂问题仍然重要：

- 不依从会使“分配的效应”与“实际接受处理的效应”分离；
- 样本流失可能重新引入选择偏差；
- 处理溢出违反无干扰假设；
- 重复查看结果或自适应停止会改变统计推断；
- 集群随机分配需要考虑集群结构的标准误；
- 意向治疗（intention-to-treat）估计目标不同于符合方案（per-protocol）效应。

**目标试验（target trial）**框架要求观测研究明确说明它试图模拟怎样的随机试验：

| 方案组成 | 必须回答的问题 |
|---|---|
| 纳入资格 | 谁进入目标总体，资格变量在何时测量？ |
| 时间零点 | 资格判断、处理分配与随访从哪个时刻对齐？ |
| 处理策略 | 处理和对照具体指什么，包括剂量、时机与宽限期？ |
| 分配机制 | 假想的随机机制是什么，模拟它需要哪些已测量变量？ |
| 随访 | 风险从何时开始和结束，如何处理删失与竞争事件？ |
| 结果 | 哪个终点、测量流程和时间范围定义成功？ |
| 估计目标 | ITT、per-protocol、ATE、ATT、生存对比，还是其他政策相关效应？ |
| 分析 | 如何处理混杂、删失、缺失、重复处理与不确定性？ |

时间零点错位尤其危险。例如，只有患者存活到真正接受治疗后才把其标记为“已处理”，就会让处理组拥有一段结果尚不可能发生的**不死时间（immortal time）**。再灵活的学习器也无法修复这种设计错误。

<details>
<summary><strong>Python：在重复研究中比较随机分配与选择性处理分配</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(44)
replications = 500
n = 1_000
true_ate = 2.0

randomized_estimates = []
observational_estimates = []

for _ in range(replications):
    x = rng.normal(size=n)
    treatment_effect = true_ate + 0.5 * x
    baseline = 1.5 * x + rng.normal(size=n)

    randomized_t = rng.binomial(1, 0.5, size=n)
    randomized_y = baseline + randomized_t * treatment_effect
    randomized_estimates.append(
        randomized_y[randomized_t == 1].mean()
        - randomized_y[randomized_t == 0].mean()
    )

    propensity = 1 / (1 + np.exp(-1.4 * x))
    observational_t = rng.binomial(1, propensity)
    observational_y = baseline + observational_t * treatment_effect
    observational_estimates.append(
        observational_y[observational_t == 1].mean()
        - observational_y[observational_t == 0].mean()
    )

print(
    "Randomized mean estimate / bias:",
    f"{np.mean(randomized_estimates):.3f}",
    f"/ {np.mean(randomized_estimates) - true_ate:.3f}",
)
print(
    "Observational mean gap / bias:",
    f"{np.mean(observational_estimates):.3f}",
    f"/ {np.mean(observational_estimates) - true_ate:.3f}",
)
print(
    "Randomized sampling SD:",
    f"{np.std(randomized_estimates, ddof=1):.3f}",
)
```

</details>

对于观测数据，一个常用识别条件是条件可交换性：

$$
(Y(1),Y(0))\perp T\mid X,
$$

并同时要求一致性与正值性：

$$
0<P(T=1\mid X=x)<1
$$

在整个目标总体上成立。可交换性无法仅凭观测数据验证；它是关于 $X$ 已经捕获相关共同原因的假设。

准实验设计会用不同假设替代条件可交换性：

| 设计 | 识别变异的来源 | 主要威胁 |
|---|---|---|
| 工具变量 | 有效工具变量会改变处理 | 排除限制、独立性、单调性与弱工具变量 |
| 回归不连续 | 分配机制在阈值处发生变化 | 阈值附近的操纵与连续性 |
| 双重差分 | 处理组与比较组的趋势构造反事实 | 平行趋势与处理时机 |
| 合成控制 | 加权供体单位近似未处理轨迹 | 供体支持与随时间变化的混杂 |

这些设计面向特定总体与效应，不能被描述成依据验证分数随意选择的可互换估计器。

#### **匹配、分层与倾向得分**

**倾向得分（propensity score）**

$$
e(X)=P(T=1\mid X)
$$

是一种平衡得分：在条件可交换性成立时，具有相同倾向得分的个体，其已测量基线协变量在处理组与对照组中具有相同分布。它并不是“处理会对该个体产生因果收益”的概率。

<div class="diagram-scroll">

![处理组与对照组的倾向得分分布需要共同支持。](assets/propensity-overlap.svg){fig-alt="处理组与对照组的倾向得分分布只在共同支持区域中重叠，区域之外的因果对比依赖外推。"}

</div>

匹配会构造明确的比较对象；分层则在倾向得分或协变量区组内比较结果。两者都需要诊断：

- 调整前后的标准化均值差异；
- 重叠区域与未匹配个体；
- 对照个体被重复使用的次数；
- 对卡钳、距离度量和是否放回匹配的敏感性；
- 尊重匹配设计的方差估计。

对于连续协变量 $X_j$，未加权标准化均值差为

$$
\operatorname{SMD}_j
=
\frac{\overline X_{1j}-\overline X_{0j}}
{\sqrt{(s_{1j}^{2}+s_{0j}^{2})/2}}.
$$

调整后应使用加权或匹配后的均值与方差替代原始组汇总。SMD 是与量纲无关的平衡诊断，而不是假设检验：样本很大时，微小失衡也可能统计显著；样本很小时，重要失衡的 $p$ 值也可能并不显著。

匹配设计会决定估计目标。让每个处理个体匹配一个或多个对照，天然面向类似 ATT 的总体；删除共同支持区外的处理个体则会改变这个目标总体。放回匹配可以改善匹配质量，却可能让少数对照代表许多处理个体，因此不确定性估计必须考虑重复使用。卡钳可以阻止明显糟糕的匹配，但卡钳过窄也可能丢弃原始总体的大部分个体。

倾向模型应根据平衡与重叠选择，而不是根据处理分类准确率选择。完美预测处理分配意味着正值性很差，并不代表建模成功。

<details>
<summary><strong>Python：基于倾向得分匹配并检查协变量平衡</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

rng = np.random.default_rng(52)
n = 6_000
x = rng.normal(size=(n, 3))
true_effect = 2.0

logit = -0.2 + 1.1 * x[:, 0] - 0.9 * x[:, 1] + 0.5 * x[:, 2]
propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, propensity)
outcome = (
    true_effect * treatment
    + 1.8 * x[:, 0]
    - 1.2 * x[:, 1]
    + 0.5 * x[:, 2] ** 2
    + rng.normal(size=n)
)

estimated_propensity = LogisticRegression(max_iter=2_000).fit(
    x, treatment
).predict_proba(x)[:, 1]

treated_idx = np.flatnonzero(treatment == 1)
control_idx = np.flatnonzero(treatment == 0)
matcher = NearestNeighbors(n_neighbors=1).fit(
    estimated_propensity[control_idx].reshape(-1, 1)
)
_, nearest = matcher.kneighbors(
    estimated_propensity[treated_idx].reshape(-1, 1)
)
matched_control_idx = control_idx[nearest[:, 0]]

def standardized_mean_difference(a, b):
    pooled_sd = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1)) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_sd

before = [
    standardized_mean_difference(x[treated_idx, j], x[control_idx, j])
    for j in range(x.shape[1])
]
after = [
    standardized_mean_difference(x[treated_idx, j], x[matched_control_idx, j])
    for j in range(x.shape[1])
]
matched_att = np.mean(outcome[treated_idx] - outcome[matched_control_idx])

balance = pd.DataFrame(
    {"covariate": ["X1", "X2", "X3"], "SMD before": before, "SMD after": after}
)
print(balance.round(3).to_string(index=False))
print(f"True ATT:      {true_effect:.3f}")
print(f"Matched ATT:   {matched_att:.3f}")
```

</details>

#### **逆概率加权**

对于二元处理，逆概率加权（inverse probability weighting, IPW）使用

$$
w_i^{\mathrm{ATE}}
=
\frac{T_i}{e(X_i)}
+
\frac{1-T_i}{1-e(X_i)}.
$$

加权后的处理组与对照组近似形成一个伪总体，其中已测量基线协变量与处理分配相互独立。

Horvitz-Thompson ATE 估计器为

$$
\widehat\tau_{\mathrm{HT}}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left[
\frac{T_iY_i}{\widehat e_i}
-
\frac{(1-T_i)Y_i}{1-\widehat e_i}
\right].
$$

Hájek 版本会分别归一化处理组与对照组权重。它在有限样本中并非严格无偏，却通常对总权重的随机波动不那么敏感。不同目标总体对应不同权重：

| 目标 | 处理组权重 | 对照组权重 | 含义 |
|---|---:|---:|---|
| ATE | $1/e(X)$ | $1/[1-e(X)]$ | 在两种处理下重构完整目标总体 |
| ATT | $1$ | $e(X)/[1-e(X)]$ | 把对照组重新加权为类似实际处理组 |
| ATC | $[1-e(X)]/e(X)$ | $1$ | 把处理组重新加权为类似实际对照组 |
| 重叠总体 | $1-e(X)$ | $e(X)$ | 强调确实存在处理不确定性的个体 |

权重诊断应包括最大值与高分位数、分处理组的权重总和、加权后的协变量平衡，以及有效样本量

$$
n_{\mathrm{eff}}
=
\frac{\left(\sum_i w_i\right)^2}{\sum_i w_i^2}.
$$

如果少数极端权重占据主导，即使原始数据有 50,000 行，加权后也可能只包含几百个平衡观测所对应的信息。

<div class="diagram-scroll">

![逆倾向权重构造平衡的伪总体。](assets/ipw-pseudopopulation.svg){fig-alt="观测个体按照处理概率的倒数加权，从而构造可比较的处理组与对照组伪总体。"}

</div>

<details>
<summary><strong>Python：使用稳定化均值估计 ATE 并诊断权重质量</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(63)
n = 10_000
x = rng.normal(size=(n, 4))
true_effect = 1.75

logit = -0.1 + 1.0 * x[:, 0] - 0.8 * x[:, 1] + 0.5 * x[:, 2]
true_propensity = 1 / (1 + np.exp(-logit))
treatment = rng.binomial(1, true_propensity)
outcome = (
    true_effect * treatment
    + 2.0 * x[:, 0]
    - 1.0 * x[:, 1]
    + 0.4 * x[:, 2] ** 2
    + rng.normal(size=n)
)

e_hat = LogisticRegression(max_iter=2_000).fit(
    x, treatment
).predict_proba(x)[:, 1]

# Positivity diagnostics are reported before any optional clipping.
raw_weights = treatment / e_hat + (1 - treatment) / (1 - e_hat)
effective_sample_size = raw_weights.sum() ** 2 / np.sum(raw_weights ** 2)

treated_weights = treatment / e_hat
control_weights = (1 - treatment) / (1 - e_hat)
hajek_ate = (
    np.sum(treated_weights * outcome) / np.sum(treated_weights)
    - np.sum(control_weights * outcome) / np.sum(control_weights)
)

print(f"True ATE:              {true_effect:.3f}")
print(f"IPW Hajek ATE:         {hajek_ate:.3f}")
print(f"Propensity range:      [{e_hat.min():.3f}, {e_hat.max():.3f}]")
print(f"Maximum ATE weight:    {raw_weights.max():.1f}")
print(f"Effective sample size: {effective_sample_size:.0f} / {n}")
```

</details>

极端权重暴露出重叠不足与不稳定外推。修剪或裁剪可以降低方差，但会改变有效目标总体并引入偏差。应同时报告处理规则和受到影响的观测比例。

#### **增广 IPW 与双重稳健估计**

结果回归估计 $\mu_t(x)=\mathbb E[Y\mid T=t,X=x]$，IPW 估计处理机制 $e(x)$。**增广逆概率加权（augmented inverse probability weighting, AIPW）**通过以下分数组合两者：

$$
\widehat\phi_i
=
\widehat\mu_1(X_i)-\widehat\mu_0(X_i)
+
\frac{T_i\{Y_i-\widehat\mu_1(X_i)\}}{\widehat e(X_i)}
-
\frac{(1-T_i)\{Y_i-\widehat\mu_0(X_i)\}}{1-\widehat e(X_i)},
$$

ATE 估计为

$$
\widehat\tau_{\mathrm{AIPW}}
=
\frac{1}{n}\sum_{i=1}^{n}\widehat\phi_i.
$$

第一项是结果模型对比；后面两个残差修正项使用观测结果纠正该对比的系统误差。在正则条件与正值性成立时，该估计器具有**双重稳健性（double robustness）**：只要结果模型或倾向模型中至少一类得到一致估计，AIPW 就是一致的。这不代表两个糟糕模型会神奇地相互抵消。

该分数同时具有 Neyman 正交性，因此可以用灵活 ML 学习足够准确的辅助函数，同时保留对低维 ATE 的一阶统计推断。有限样本中，辅助预测应使用折外预测：每个观测的倾向得分与潜在结果预测都来自没有使用该观测训练的模型。

<details>
<summary><strong>Python：实现交叉拟合 AIPW，并检查双重稳健性</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(64)
n = 12_000
x = rng.normal(size=(n, 5))

# The propensity belongs to the logistic model, while the outcome surface
# is nonlinear. The treatment effect is heterogeneous but averages to 1.5.
true_logit = -0.2 + 0.8 * x[:, 0] - 0.7 * x[:, 1] + 0.4 * x[:, 2]
true_propensity = 1 / (1 + np.exp(-true_logit))
treatment = rng.binomial(1, true_propensity)
mu_0 = (
    1.0
    + np.sin(x[:, 0])
    + 0.8 * x[:, 1] ** 2
    - 0.5 * x[:, 2] * x[:, 3]
)
true_cate = 1.5 + 0.4 * x[:, 0]
outcome = mu_0 + treatment * true_cate + rng.normal(size=n)

e_correct = np.empty(n)
e_poor = np.empty(n)
mu0_flexible = np.empty(n)
mu1_flexible = np.empty(n)
mu0_poor = np.empty(n)
mu1_poor = np.empty(n)

kfold = KFold(n_splits=4, shuffle=True, random_state=64)
for fold, (train_idx, test_idx) in enumerate(kfold.split(x)):
    x_train, x_test = x[train_idx], x[test_idx]
    t_train, y_train = treatment[train_idx], outcome[train_idx]

    propensity_model = LogisticRegression(max_iter=2_000).fit(
        x_train, t_train
    )
    e_correct[test_idx] = propensity_model.predict_proba(x_test)[:, 1]
    e_poor[test_idx] = DummyClassifier(strategy="prior").fit(
        x_train, t_train
    ).predict_proba(x_test)[:, 1]

    for t_value, flexible_target, poor_target in [
        (0, mu0_flexible, mu0_poor),
        (1, mu1_flexible, mu1_poor),
    ]:
        in_group = t_train == t_value
        flexible_model = HistGradientBoostingRegressor(
            max_iter=180,
            max_leaf_nodes=31,
            min_samples_leaf=30,
            l2_regularization=1.0,
            random_state=10 * fold + t_value,
        ).fit(x_train[in_group], y_train[in_group])
        poor_model = DummyRegressor(strategy="mean").fit(
            x_train[in_group], y_train[in_group]
        )
        flexible_target[test_idx] = flexible_model.predict(x_test)
        poor_target[test_idx] = poor_model.predict(x_test)

def aipw_summary(mu1_hat, mu0_hat, e_hat):
    # Clipping here only guards numerical explosions in the demonstration;
    # a real analysis must report overlap and any changed target population.
    e_hat = np.clip(e_hat, 0.02, 0.98)
    plugin = np.mean(mu1_hat - mu0_hat)
    ipw = np.mean(
        treatment * outcome / e_hat
        - (1 - treatment) * outcome / (1 - e_hat)
    )
    score = (
        mu1_hat
        - mu0_hat
        + treatment * (outcome - mu1_hat) / e_hat
        - (1 - treatment) * (outcome - mu0_hat) / (1 - e_hat)
    )
    estimate = score.mean()
    standard_error = score.std(ddof=1) / np.sqrt(n)
    return plugin, ipw, estimate, standard_error

configurations = [
    ("both informative", mu1_flexible, mu0_flexible, e_correct),
    ("outcome informative only", mu1_flexible, mu0_flexible, e_poor),
    ("propensity informative only", mu1_poor, mu0_poor, e_correct),
    ("both poor", mu1_poor, mu0_poor, e_poor),
]

rows = []
for label, mu1_hat, mu0_hat, e_hat in configurations:
    plugin, ipw, aipw, se = aipw_summary(mu1_hat, mu0_hat, e_hat)
    rows.append(
        {
            "nuisance models": label,
            "plugin": plugin,
            "IPW": ipw,
            "AIPW": aipw,
            "AIPW SE": se,
        }
    )

print(f"True sample ATE: {true_cate.mean():.3f}")
print(pd.DataFrame(rows).round(3).to_string(index=False))
```

</details>

交叉拟合分数的经验标准差除以 $\sqrt n$，可以在独立抽样与正则条件下给出影响函数标准误。集群分配、重复观测、调查抽样或时间依赖需要尊重相应依赖结构的方差估计器。由此得到的区间只度量**以识别假设成立为条件的抽样不确定性**，并不包含隐藏混杂带来的不确定性。


### **因果机器学习估计器**

因果机器学习使用灵活预测模型估计辅助函数，例如

$$
\mu_t(x)=\mathbb E[Y\mid T=t,X=x]
\quad\text{and}\quad
e(x)=P(T=1\mid X=x),
$$

或者直接建模异质处理效应。这些算法提高了函数形式的灵活性，但不会让隐藏混杂、一致性、正值性或抽样假设消失。

#### **Uplift 与 Meta-Learner**

**Uplift modeling** 按照增量响应而不是预测结果对个体排序。一个用户在处理和对照条件下都具有很高的购买概率，因此可能是优秀的预测目标，但其 uplift 可能接近于零。

Meta-learner 通过不同方式组织标准监督学习器：

- **S-learner**：拟合一个模型 $\hat\mu(x,t)$，预测时切换处理值；
- **T-learner**：分别拟合 $\hat\mu_1(x)$ 和 $\hat\mu_0(x)$；
- **X-learner**：在每个组中插补处理效应，拟合效应模型，再使用倾向信息组合模型；
- **R-learner**：对处理和结果进行残差化，再从正交化损失中学习效应函数。

<div class="diagram-scroll">

![S-、T- 与 X-learner 以不同方式组织监督结果模型与效应模型。](assets/causal-meta-learners.svg){fig-alt="S-learner 使用一个结果模型，T-learner 使用相互独立的处理组与对照组模型，X-learner 则插补并组合处理效应。"}

</div>

<details>
<summary><strong>Python：在已知异质效应上比较 S-、T- 与 X-learner</strong></summary>

```python
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(71)
n = 8_000
x = rng.normal(size=(n, 4))
treatment = rng.binomial(1, 0.5, size=n)

mu_0 = 1.0 + x[:, 0] ** 2 - 0.5 * x[:, 1] + np.sin(x[:, 2])
true_cate = 1.0 + 0.8 * x[:, 0] + 0.7 * (x[:, 1] > 0)
outcome = mu_0 + treatment * true_cate + rng.normal(scale=1.0, size=n)

x_train, x_test, t_train, _, y_train, _, _, tau_test = train_test_split(
    x, treatment, outcome, true_cate, test_size=0.35, random_state=71
)

def forest(seed):
    return RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=15,
        random_state=seed,
        n_jobs=-1,
    )

# S-learner
s_model = forest(1).fit(np.column_stack([x_train, t_train]), y_train)
s_tau = (
    s_model.predict(np.column_stack([x_test, np.ones(len(x_test))]))
    - s_model.predict(np.column_stack([x_test, np.zeros(len(x_test))]))
)

# T-learner
model_0 = forest(2).fit(x_train[t_train == 0], y_train[t_train == 0])
model_1 = forest(3).fit(x_train[t_train == 1], y_train[t_train == 1])
t_tau = model_1.predict(x_test) - model_0.predict(x_test)

# X-learner with randomized propensity e(x)=0.5
mu0_for_treated = model_0.predict(x_train[t_train == 1])
mu1_for_control = model_1.predict(x_train[t_train == 0])
d_treated = y_train[t_train == 1] - mu0_for_treated
d_control = mu1_for_control - y_train[t_train == 0]
tau_1 = forest(4).fit(x_train[t_train == 1], d_treated)
tau_0 = forest(5).fit(x_train[t_train == 0], d_control)
x_tau = 0.5 * tau_0.predict(x_test) + 0.5 * tau_1.predict(x_test)

def rmse(estimate):
    return np.sqrt(np.mean((estimate - tau_test) ** 2))

print(f"S-learner CATE RMSE: {rmse(s_tau):.3f}")
print(f"T-learner CATE RMSE: {rmse(t_tau):.3f}")
print(f"X-learner CATE RMSE: {rmse(x_tau):.3f}")
print("One simulation is not a universal algorithm ranking.")
```

</details>

#### **异质效应与策略评估**

CATE 很难评估，因为个体的 $Y_i(1)-Y_i(0)$ 永远不可观测。真实数据中无法直接计算普通 RMSE；而且一个模型即使能准确预测 $Y$，也可能完全无法正确排列处理效应。评估应区分三个问题：

| 评估目标 | 核心问题 | 较有依据的证据 |
|---|---|---|
| 校准 | 被预测为效应接近 $a$ 的群体，平均真实效应是否也接近 $a$？ | 在诚实分组中比较预测效应与试验或双重稳健效应估计 |
| 排序 | 模型是否把更响应处理的个体排在较不响应者之前？ | Uplift/Qini 曲线、RATE 或 AUTOC 汇总，以及留出样本上的不确定性 |
| 决策价值 | 在考虑处理成本与约束后，使用分数的策略是否提高期望效用？ | 离策略价值估计或前瞻性随机策略试验 |

合成与半合成基准可以观察真实 CATE，适合调试，却可能奖励现实数据并不满足的结构。在处理概率已知为 $e$ 的随机留出数据上，变换结果

$$
\Gamma_i
=
\frac{T_iY_i}{e}
-
\frac{(1-T_i)Y_i}{1-e}
$$

满足 $\mathbb E[\Gamma_i\mid X_i=x]=\tau(x)$。它在个体层面噪声很大，因此应在足够大且预先规定的分数组内求平均，而不能当作个体标签使用。对于观测数据，通常应使用交叉拟合的 AIPW 伪结果。

对于二元策略 $\pi(X)\in\{0,1\}$ 和已知处理倾向，其价值可估计为

$$
\widehat V(\pi)
=
\frac{1}{n}
\sum_{i=1}^{n}
\frac{\mathbb I\{T_i=\pi(X_i)\}Y_i}
{P(T_i\mid X_i)}.
$$

如果处理有成本，应从处理潜在结果中扣除成本，或在比较策略之前把成本纳入效用。策略应在一个样本上学习，再在独立样本上评估，或者采用嵌套交叉拟合。

<details>
<summary><strong>Python：在随机留出集上评估 CATE 校准与策略价值</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(75)

def make_randomized_data(n):
    x = rng.normal(size=(n, 4))
    treatment = rng.binomial(1, 0.5, size=n)
    mu0 = 1.0 + x[:, 0] ** 2 - 0.5 * x[:, 1] + np.sin(x[:, 2])
    cate = 0.3 + 1.0 * np.tanh(x[:, 0]) + 0.6 * (x[:, 1] > 0)
    outcome = mu0 + treatment * cate + rng.normal(scale=1.0, size=n)
    return x, treatment, outcome, mu0, cate

x_train, t_train, y_train, _, _ = make_randomized_data(8_000)
x_test, t_test, y_test, mu0_test, cate_test = make_randomized_data(12_000)

def forest(seed):
    return RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=25,
        random_state=seed,
        n_jobs=-1,
    )

# Fit a T-learner only on the training sample.
model_0 = forest(1).fit(x_train[t_train == 0], y_train[t_train == 0])
model_1 = forest(2).fit(x_train[t_train == 1], y_train[t_train == 1])
cate_hat = model_1.predict(x_test) - model_0.predict(x_test)

# With randomized e=0.5, this noisy pseudo-outcome has conditional mean CATE.
transformed_outcome = (
    t_test * y_test / 0.5
    - (1 - t_test) * y_test / 0.5
)

calibration = pd.DataFrame(
    {
        "predicted CATE": cate_hat,
        "experimental effect": transformed_outcome,
        "true CATE (simulation only)": cate_test,
    }
)
calibration["score decile"] = pd.qcut(
    calibration["predicted CATE"], 10, labels=False
)
calibration_table = calibration.groupby("score decile").mean()

# Treat only when estimated benefit exceeds treatment cost.
treatment_cost = 0.40
policy = (cate_hat > treatment_cost).astype(int)
observed_net_outcome = y_test - treatment_cost * t_test

def randomized_policy_value(policy_action):
    # The indicator keeps outcomes from units whose randomized action agrees
    # with the policy; division by 0.5 reconstructs the full population.
    return np.mean(
        (t_test == policy_action) * observed_net_outcome / 0.5
    )

estimated_policy_value = randomized_policy_value(policy)
estimated_treat_all = randomized_policy_value(np.ones(len(policy), dtype=int))
estimated_treat_none = randomized_policy_value(np.zeros(len(policy), dtype=int))
true_policy_value = np.mean(mu0_test + policy * (cate_test - treatment_cost))

print(calibration_table.round(3).to_string())
print(f"\nEstimated learned-policy value: {estimated_policy_value:.3f}")
print(f"True learned-policy value:      {true_policy_value:.3f}")
print(f"Estimated treat-all value:      {estimated_treat_all:.3f}")
print(f"Estimated treat-none value:     {estimated_treat_none:.3f}")
```

</details>

<div class="diagram-scroll">

![EconML 的双重稳健 CATE 解释器使用浅层树概括估计的效应异质性。](assets/econml-dr-cate-tree.png){fig-alt="浅层解释树把总体划分为具有不同平均 CATE 估计、不确定性汇总和样本量的子群体。" width="95%"}

</div>

*示例来自官方 [EconML 仓库](https://github.com/py-why/EconML/blob/main/notebooks/images/dr_cate_tree.png)，并依据仓库的 [MIT 许可证](https://github.com/py-why/EconML/blob/main/LICENSE)分发。这棵浅层树是底层 CATE 模型的解释工具，并不能证明用于分裂的变量本身导致结果。*

校准、排序与策略价值可能彼此不一致。模型可能正确排列效应，却把效应幅度整体收缩；这可能保留 top-$k$ 策略，却错误判断成本阈值。反过来，全局校准的模型也可能遗漏有用异质性。应报告不确定性、重叠性、子群体规模，以及不同诚实分割下的稳定性，而不是在看到估计效应后才挑选最漂亮的子群体。

#### **双重机器学习**

考虑部分线性模型

$$
Y=\theta_0 T+g_0(X)+\zeta,
\qquad
T=m_0(X)+V,
$$

其中 $\mathbb E[\zeta\mid T,X]=0$ 且 $\mathbb E[V\mid X]=0$。定义

$$
\ell_0(X)=\mathbb E[Y\mid X]
=
\theta_0m_0(X)+g_0(X).
$$

直接代入的回归可能继承灵活估计 $\ell_0$ 和 $m_0$ 时产生的正则化偏差。双重/去偏机器学习（double/debiased machine learning, DML）使用正交分数

$$
\psi(W;\theta,\eta)
=
\left(T-m(X)\right)
\left[
Y-\ell(X)-\theta\left(T-m(X)\right)
\right].
$$

目标参数满足 $\mathbb E[\psi(W;\theta_0,\eta_0)]=0$。Neyman 正交性表示期望分数对辅助函数的局部扰动不敏感：

$$
\left.
\frac{\partial}{\partial r}
\mathbb E\!\left[
\psi\!\left(W;\theta_0,\eta_0+r(\eta-\eta_0)\right)
\right]
\right|_{r=0}
=0.
$$

这会消除一阶正则化偏差，但辅助函数误差的二阶乘积仍然存在。**交叉拟合（cross-fitting）**避免使用同一个观测既拟合高容量辅助模型、又评估其残差：

1. 把观测划分为 $K$ 折；
2. 在不包含第 $k$ 折的数据上拟合 $\widehat\ell^{(-k)}$ 与 $\widehat m^{(-k)}$；
3. 对第 $k$ 折预测，并构造 $\widetilde Y_i=Y_i-\widehat\ell^{(-k)}(X_i)$ 和 $\widetilde T_i=T_i-\widehat m^{(-k)}(X_i)$；
4. 汇总所有折外残差，求解 $\sum_i\widetilde T_i(\widetilde Y_i-\theta\widetilde T_i)=0$。

<div class="diagram-scroll">

![双重机器学习先构造折外处理残差与结果残差，再估计因果参数。](assets/double-ml-cross-fitting.svg){fig-alt="结果模型与处理模型在不同数据折上训练，构造折外残差，再通过正交分数估计处理效应。"}

</div>

<details>
<summary><strong>Python：实现交叉拟合的残差对残差 DML</strong></summary>

```python
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

rng = np.random.default_rng(82)
n = 10_000
p = 8
x = rng.normal(size=(n, p))
theta_true = 2.25

m_x = 0.8 * np.sin(x[:, 0]) + 0.5 * x[:, 1] * x[:, 2] - 0.3 * x[:, 3] ** 2
g_x = 2.0 * m_x + 1.2 * np.cos(x[:, 0]) + x[:, 1] ** 2 + 0.7 * x[:, 4] * x[:, 5]
treatment = m_x + rng.normal(size=n)
outcome = theta_true * treatment + g_x + rng.normal(scale=2.0, size=n)

outcome_residual = np.empty(n)
treatment_residual = np.empty(n)
kfold = KFold(n_splits=5, shuffle=True, random_state=82)

for train_idx, test_idx in kfold.split(x):
    model_y = HistGradientBoostingRegressor(
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        random_state=1,
    ).fit(x[train_idx], outcome[train_idx])
    model_t = HistGradientBoostingRegressor(
        max_iter=250,
        max_leaf_nodes=31,
        min_samples_leaf=25,
        l2_regularization=1.0,
        random_state=2,
    ).fit(x[train_idx], treatment[train_idx])

    outcome_residual[test_idx] = outcome[test_idx] - model_y.predict(x[test_idx])
    treatment_residual[test_idx] = (
        treatment[test_idx] - model_t.predict(x[test_idx])
    )

dml_theta = np.dot(treatment_residual, outcome_residual) / np.dot(
    treatment_residual, treatment_residual
)

# The normalized orthogonal score is an estimated influence function for
# theta under iid sampling in this partially linear model.
moment_residual = treatment_residual * (
    outcome_residual - dml_theta * treatment_residual
)
influence = moment_residual / np.mean(treatment_residual ** 2)
standard_error = influence.std(ddof=1) / np.sqrt(n)
confidence_interval = (
    dml_theta - 1.96 * standard_error,
    dml_theta + 1.96 * standard_error,
)

naive_theta = LinearRegression().fit(
    treatment.reshape(-1, 1), outcome
).coef_[0]

print(f"True theta:       {theta_true:.3f}")
print(f"Naive regression: {naive_theta:.3f}")
print(f"Cross-fitted DML: {dml_theta:.3f}")
print(f"Influence SE:     {standard_error:.3f}")
print(
    "Approximate 95% CI:",
    f"[{confidence_interval[0]:.3f}, {confidence_interval[1]:.3f}]",
)
```

</details>

分数必须匹配估计目标与处理结构。上面的残差对残差分数面向常数部分线性系数，常用于连续处理；对于二元处理的 ATE，交互回归模型分数就是上一节的 AIPW 分数；对于异质效应，DML 可以先正交化辅助函数，再拟合最终 CATE 模型。

正交性降低了结果对辅助函数估计的敏感度，但不会移除遗漏混杂或正值性失败。有效的不确定性估计还要求正则条件、辅助函数具有足够快的收敛速度、按集群或时间结构划分数据折，并且估计目标与所选分数相匹配。如果不断依据最终因果估计调节辅助模型，可能会破坏交叉拟合试图建立的分离。

#### **Causal Forest**

Causal forest 估计

$$
\tau(x)=\mathbb E[Y(1)-Y(0)\mid X=x]
$$

它构造的树邻域面向处理效应异质性，而不是只面向结果预测。现代实现通常组合：

- **诚实分裂（honest splitting）**：一个子样本选择树结构，另一个子样本估计叶节点效应；
- 局部处理中心化与结果中心化；
- 面向处理效应差异的分裂准则；
- 子采样与森林权重；
- 在明确条件下得到的渐近方差或自助法式不确定性。

一种有用的理解方式是：森林定义了自适应邻域权重 $\alpha_i(x)$，经常与 $x$ 落入同一叶节点的训练观测会获得更大权重。使用交叉拟合的结果与倾向辅助函数后，局部效应可以通过下式求解：

$$
\sum_i
\alpha_i(x)
\{T_i-\widehat e(X_i)\}
\left[
Y_i-\widehat m(X_i)
-\tau(x)\{T_i-\widehat e(X_i)\}
\right]
=0.
$$

这不同于直接对观测结果拟合随机森林，再把两次预测相减。这里的分裂、残差化、叶节点支持和诚实性都围绕局部因果矩条件设计，而不只是围绕预测纯度设计。

<div class="diagram-scroll">

![Causal forest 使用诚实建树与局部处理效应估计。](assets/causal-forest-honesty.svg){fig-alt="Causal forest 分离结构样本与估计样本，构建寻找处理效应异质性的树，再把局部邻域聚合为 CATE 估计。"}

</div>

<details>
<summary><strong>Python：构造教学型 honest causal forest</strong></summary>

```python
import numpy as np
from sklearn.tree import DecisionTreeRegressor

rng = np.random.default_rng(93)
n_train = 4_000
n_test = 1_500

def make_data(n):
    x = rng.normal(size=(n, 5))
    treatment = rng.binomial(1, 0.5, size=n)
    cate = 1.0 + 0.9 * np.tanh(x[:, 0]) - 0.6 * (x[:, 1] > 0)
    baseline = x[:, 0] ** 2 + 0.5 * x[:, 2] - 0.4 * x[:, 3] * x[:, 4]
    outcome = baseline + treatment * cate + rng.normal(scale=1.0, size=n)
    return x, treatment, outcome, cate

x_train, t_train, y_train, _ = make_data(n_train)
x_test, _, _, tau_test = make_data(n_test)

# This compact implementation illustrates honesty:
# one subsample discovers heterogeneous leaves, while a disjoint subsample
# estimates treated-control differences inside those leaves.
n_trees = 250
tree_predictions = np.empty((n_test, n_trees))

for tree_id in range(n_trees):
    subsample = rng.choice(n_train, size=int(0.75 * n_train), replace=False)
    rng.shuffle(subsample)
    midpoint = len(subsample) // 2
    structure_idx = subsample[:midpoint]
    estimation_idx = subsample[midpoint:]

    # With randomized propensity 0.5, this transformed outcome has
    # conditional expectation E[Z|X=x] = tau(x).
    transformed_outcome = (
        2 * (2 * t_train[structure_idx] - 1) * y_train[structure_idx]
    )
    tree = DecisionTreeRegressor(
        max_depth=6,
        min_samples_leaf=45,
        random_state=tree_id,
    ).fit(x_train[structure_idx], transformed_outcome)

    estimation_leaves = tree.apply(x_train[estimation_idx])
    test_leaves = tree.apply(x_test)
    global_effect = (
        y_train[estimation_idx][t_train[estimation_idx] == 1].mean()
        - y_train[estimation_idx][t_train[estimation_idx] == 0].mean()
    )

    leaf_effect = {}
    for leaf in np.unique(estimation_leaves):
        in_leaf = estimation_leaves == leaf
        leaf_t = t_train[estimation_idx][in_leaf]
        leaf_y = y_train[estimation_idx][in_leaf]
        if np.sum(leaf_t == 1) >= 10 and np.sum(leaf_t == 0) >= 10:
            leaf_effect[leaf] = (
                leaf_y[leaf_t == 1].mean() - leaf_y[leaf_t == 0].mean()
            )
        else:
            leaf_effect[leaf] = global_effect

    tree_predictions[:, tree_id] = np.array(
        [leaf_effect.get(leaf, global_effect) for leaf in test_leaves]
    )

cate_hat = tree_predictions.mean(axis=1)

rmse = np.sqrt(np.mean((cate_hat - tau_test) ** 2))
correlation = np.corrcoef(cate_hat, tau_test)[0, 1]

print(f"True test ATE:       {tau_test.mean():.3f}")
print(f"Estimated test ATE:  {cate_hat.mean():.3f}")
print(f"CATE RMSE:           {rmse:.3f}")
print(f"CATE rank correlation: {correlation:.3f}")
```

</details>

这个紧凑实现展示了样本分割和异质叶节点，但不能替代生产级 generalized random forest 或正交 causal forest 软件；后者还提供辅助函数残差化、专用分裂准则、有效方差估计与更完整的边界情况处理。当叶节点很小，或者调参与评估重复使用同一份数据时，causal forest 可能从噪声中发现虚假异质性。应报告重叠性、校准、子群体稳定性、置信区间、策略价值，以及不同诚实分割或随机种子下的结果。变量重要性并不是该特征本身的因果效应。


### **因果发现**

因果效应估计从一个假定的图或研究设计出发。**因果发现（causal discovery）**询问的是：能否从观测或干预数据中学习图结构的某些部分。得到的答案通常是一个等价类，而不是唯一确定的 DAG。

典型假设包括：

- 因果 Markov 性质；
- 忠实性，使条件独立关系能够反映图分离；
- 因果充分性，或一个显式潜变量模型；
- 独立同分布样本，除非明确建模时间结构；
- 不存在未建模的选择偏差；
- 可靠的条件独立检验或图评分；
- DAG 方法所要求的无环性。

违反这些假设可能反转已有边，或凭空产生错误方向。

#### **基于约束与基于评分的方法**

**PC 算法**基于约束：

1. 从稠密无向图开始；
2. 当条件独立检验找到分离集时移除边；
3. 定向无屏蔽碰撞结构；
4. 使用定向规则传播方向，同时避免形成环和没有依据的碰撞结构。

**Greedy Equivalence Search（GES）**基于评分。它在 Markov 等价类之间搜索，通过添加边、再删除边来改善 BIC 等带复杂度惩罚的似然评分。

当潜在共同原因可能存在时，**Fast Causal Inference（FCI）**扩展了基于约束的推理，并返回部分祖先图，而不是假装所有共同原因都已测量。其他方法通过额外函数假设获得方向信息，例如 LiNGAM 的非高斯噪声或加性噪声模型的方向不对称；NOTEARS 等可微方法则用连续无环约束替代组合搜索。这些假设提供了额外信息，同时也缩小了结论有效的数据生成过程范围。

| 方法类别 | 主要信号 | 典型输出 | 重要假设或弱点 |
|---|---|---|---|
| PC | 条件独立检验 | CPDAG | 通常要求因果充分性、忠实性与可靠检验 |
| FCI | 允许潜变量时的条件独立关系 | PAG | 依赖忠实性；条件集合很多时昂贵且不稳定 |
| GES | 带惩罚似然或其他可分解评分 | CPDAG | 评分一致性与搜索质量 |
| LiNGAM / 加性噪声方法 | 分布或函数不对称 | 定向更完整的 DAG | 函数与噪声族设定正确 |
| NOTEARS 类优化 | 平滑拟合与无环约束 | 加权 DAG | 优化、模型族、阈值与无环假设 |
| 时间序列发现 | 滞后依赖与时间顺序 | 滞后或同期图 | 平稳性、采样频率与自相关模型 |

**CPDAG** 表示一个 DAG 的 Markov 等价类：有向边在等价类中的每个 DAG 里方向相同，无向边则无法仅依据观测条件独立关系确定方向。如果两个 DAG 具有相同骨架与相同无屏蔽碰撞结构，它们就 Markov 等价。**PAG** 表示允许潜在混杂或选择机制时的等价类；圆端点、尾端与箭头端编码哪些祖先关系尚未确定或在所有候选图中保持不变。

<div class="diagram-scroll">

![基于约束与基于评分的因果发现，在假设下返回图或等价类。](assets/causal-discovery-equivalence.svg){fig-alt="基于约束的发现通过条件独立关系移除边，基于评分的发现优化图拟合与复杂度，两者通常都只能识别等价类，而不是唯一 DAG。"}

</div>

三节点链与叉形结构可以属于同一个 Markov 等价类：

$$
X\rightarrow M\rightarrow Y,\qquad
X\leftarrow M\rightarrow Y,\qquad
X\leftarrow M\leftarrow Y
$$

它们都蕴含 $X\perp Y\mid M$。相反，碰撞结构 $X\rightarrow M\leftarrow Y$ 蕴含边际独立，但在调整 $M$ 后变为条件相关。

<details>
<summary><strong>Python：检查叉形、链式与碰撞结构的条件独立特征</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(104)
n = 50_000

def correlation(a, b):
    return np.corrcoef(a, b)[0, 1]

def partial_correlation(a, b, conditioning):
    z = np.asarray(conditioning).reshape(n, -1)
    a_residual = a - LinearRegression().fit(z, a).predict(z)
    b_residual = b - LinearRegression().fit(z, b).predict(z)
    return correlation(a_residual, b_residual)

# Fork: C -> T and C -> Y
c = rng.normal(size=n)
t_fork = 0.9 * c + rng.normal(size=n)
y_fork = -0.8 * c + rng.normal(size=n)

# Chain: X -> M -> Y
x_chain = rng.normal(size=n)
m_chain = 0.9 * x_chain + rng.normal(size=n)
y_chain = 0.8 * m_chain + rng.normal(size=n)

# Collider: X -> S <- Y
x_collider = rng.normal(size=n)
y_collider = rng.normal(size=n)
s_collider = x_collider + y_collider + rng.normal(scale=0.5, size=n)

print(
    "Fork corr(T,Y), partial corr(T,Y|C):",
    f"{correlation(t_fork, y_fork):.3f}",
    f"{partial_correlation(t_fork, y_fork, c):.3f}",
)
print(
    "Chain corr(X,Y), partial corr(X,Y|M):",
    f"{correlation(x_chain, y_chain):.3f}",
    f"{partial_correlation(x_chain, y_chain, m_chain):.3f}",
)
print(
    "Collider corr(X,Y), partial corr(X,Y|S):",
    f"{correlation(x_collider, y_collider):.3f}",
    f"{partial_correlation(x_collider, y_collider, s_collider):.3f}",
)
```

</details>

仅靠条件独立无法确定所有边的方向。时间顺序、干预、多个环境、非高斯假设、加性噪声结构或领域知识可以提供额外定向信息。

有限样本错误可能沿整张图传播。一次错误的独立性判断就可能删除真实邻接关系、制造错误分离集、定向错误碰撞结构，并触发后续更多错误方向。因此应在不同检验水平、自助样本、变量集合与合理领域约束下比较结果。稳定性是有用证据，但在同一组错误假设下稳定的图仍然可能是错的。

因果发现与效应估计是两个不同任务。边 $X\rightarrow Y$ 不会给出因果效应大小；而部分定向等价类中可能包含要求不同调整集合的图。发现得到的图应被视为需要结合时间知识、实验与领域审核来压力测试的假说，而不是发布无条件因果结论的许可证。


### **假设、敏感性与失败模式**

因果假设不是附录，而是因果主张的定义。一个实践分析至少应明确说明：

| 假设或设计条件 | 它保护什么 | 典型失败 |
|---|---|---|
| 定义明确的处理与一致性 | 可解释的潜在结果 | 存在多个处理版本，或实施方式不断变化 |
| 可交换性 / 无未测量混杂 | 调整后的组间可比性 | 隐藏的严重程度、意图、资格或选择机制 |
| 正值性 / 重叠性 | 因果对比具有经验支持 | 处理分配接近确定性 |
| 正确的时间顺序 | 只调整处理前变量 | 使用处理后特征或结果泄漏 |
| 无干扰 | 个体层面的潜在结果 | 网络、市场、传染病或共享资源效应 |
| 准确测量 | 正确的调整变量与结果 | 差异性测量误差或代理混杂 |
| 正确抽样与效应迁移 | 与目标总体相关 | 试验志愿者不同于部署总体 |

<div class="diagram-scroll">

![DoWhy 工作流区分因果建模、识别、估计与反驳。](assets/dowhy-causal-workflow.png){fig-alt="DoWhy 工作流从输入数据与因果图开始，识别目标估计量、估计因果效应，并对估计进行反驳或压力测试。" width="92%"}

</div>

*工作流图片来自官方 [DoWhy 仓库](https://github.com/py-why/dowhy/blob/main/docs/images/dowhy-schematic.png)，并依据仓库的 [MIT 许可证](https://github.com/py-why/dowhy/blob/main/LICENSE)分发。这里的“反驳”是寻找估计脆弱性的证据；通过反驳检验并不能证明因果假设成立。*

可观测诊断包括协变量平衡、倾向得分重叠、权重尾部、有效样本量、残差模式、处理前趋势、安慰剂结果与子群体稳定性。它们可以暴露失败，却不能证明不存在隐藏混杂。

不同检查回答不同问题：

| 工具 | 它可以暴露什么 | 它无法证明什么 |
|---|---|---|
| 平衡与重叠诊断 | 实施后的设计是否平衡已测量协变量并保留共同支持 | 是否遗漏重要混杂变量 |
| 残差与辅助模型检查 | 严重函数误设和不稳定预测 | 正确因果方向或调整集合 |
| 安慰剂与负对照检验 | 某些时间、选择或隐藏偏差矛盾 | 不存在所有可能的隐藏偏差机制 |
| 样本分割与重复随机种子 | 过拟合与子群体不稳定 | 因果识别 |
| 置信区间 | 给定模型与设计下的抽样变异 | 不可检验因果假设的不确定性 |
| 敏感性分析 | 指定违反需要多强才会改变结论 | 违反机制的真实强度或形式 |

应区分**统计不确定性**与**识别不确定性**。普通 95% 置信区间询问的是：如果研究设计、估计目标、测量与识别假设都正确，重复抽样时估计器会怎样变化。它并没有为“没有未测量混杂”赋予 95% 概率。第二层不确定性需要通过敏感性范围、界与替代因果模型表达。

敏感性分析询问的是：当明确假设被不同程度地违反时，结论会如何变化：

- 用于匹配研究的 Rosenbaum 式隐藏偏差参数；
- 风险比尺度上的 E-value 或偏差因子；
- 描述遗漏变量强度的偏 $R^2$ 或稳健性值；
- 负对照暴露或负对照结果；
- 安慰剂干预与时间上不可能出现的效应；
- 在有界混杂或缺失机制下得到的界；
- 替代调整变量集合与测量假设。

<details>
<summary><strong>Python：在指定隐藏混杂模型下绘制处理效应偏差</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(115)
n = 40_000
theta_true = 1.5
strengths = [0.0, 0.3, 0.6, 0.9]
rows = []

for effect_on_treatment in strengths:
    for effect_on_outcome in strengths:
        x = rng.normal(size=n)
        hidden_u = rng.normal(size=n)
        treatment = (
            0.8 * x
            + effect_on_treatment * hidden_u
            + rng.normal(size=n)
        )
        outcome = (
            theta_true * treatment
            + 1.2 * x
            + effect_on_outcome * hidden_u
            + rng.normal(size=n)
        )

        # The analyst adjusts for X but cannot observe U.
        estimate = LinearRegression().fit(
            np.column_stack([treatment, x]), outcome
        ).coef_[0]
        rows.append(
            {
                "U->T": effect_on_treatment,
                "U->Y": effect_on_outcome,
                "estimated effect": estimate,
                "bias": estimate - theta_true,
            }
        )

result = pd.DataFrame(rows)
bias_table = result.pivot(index="U->T", columns="U->Y", values="bias")
print("Bias after adjusting for observed X")
print(bias_table.round(3).to_string())
```

</details>

这张表不是隐藏混杂检验。它回答的是一个条件问题：如果遗漏变量以指定强度和函数形式影响处理与结果，偏差会有多大？敏感性结论的意义不会超过所设定的违反模型本身。

常见因果机器学习失败模式包括：

- 使用预测误差优化因果估计目标；
- 根据特征重要性选择调整变量；
- 以媒介变量或碰撞变量为条件；
- 在某个处理组几乎没有支持时分别拟合处理模型；
- 把 CATE 排名解释为已知的个体效应；
- 使用发现子群体时已经见过的结果进行调参；
- 报告很窄的模型置信区间，却忽略识别不确定性；
- 在没有建模总体差异的情况下迁移因果效应；
- 把因果发现输出当作真实因果图。


### **因果推理何时会改变机器学习问题**

当模型输出将指导行动，而且目标是该行动的**增量后果**时，就需要因果推理。它会改变以下应用中的学习目标：

- 处理分配与个性化医疗；
- 定价、促销与用户留存触达；
- 政策评估与资源分配；
- 带有反馈循环的推荐系统；
- 诊断应该改变哪个系统组件；
- 估计领域变化或政策变化下的效应。

<div class="diagram-scroll">

![因果建模从干预问题推进到设计、估计目标、识别与估计。](assets/causal-formulation-decision.svg){fig-alt="工作流首先定义干预问题，然后评估研究设计，选择估计目标，建立识别论证，最后才选择估计器。"}

</div>

应采用研究设计能够支持的最弱方法：

| 情形 | 主要目标 | 合适的起点 |
|---|---|---|
| 只关心未来结果，行动不会改变数据生成过程 | 预测风险 | 监督学习与对部署有效的评估 |
| 随机二元处理，平均效应已经足够 | ATE / ITT | 均值差与精度调整 |
| 观测处理，已测量混杂变量假设可以辩护 | ATE、ATT 或 CATE | 结果回归、加权、匹配或双重稳健估计 |
| 高维辅助函数、低维因果参数 | 结构系数或 ATE | 交叉拟合 DML / 正交分数 |
| 处理效应可能变化，而且重叠充分 | CATE / 策略价值 | Meta-learner、R-learner、causal forest |
| 分配遵循阈值、工具变量或政策时间 | 设计特定的局部效应 | RD、IV、DiD 或合成控制 |
| 因果图本身不确定 | 等价类或因果假说 | 因果发现、干预与领域审核 |

接受一个因果估计前，应记录：

1. 目标试验：资格、时间零点、处理策略、随访、结果与估计目标；
2. 因果图或潜在结果假设；
3. 为什么该效应可以被识别；
4. 哪些观测提供了重叠支持；
5. 辅助模型和超参数如何进行交叉拟合；
6. 来自抽样与模型拟合的不确定性；
7. 对隐藏混杂、测量、缺失与干扰的敏感性；
8. 效应能否迁移到部署总体；
9. 使用该估计的策略将如何被安全评估。

预测回答“在当前系统下，什么最可能发生？”因果推断回答“在一个指定干预下，什么会发生变化？”当灵活预测支持第二个问题，同时没有取代使该问题可回答的假设时，因果机器学习才真正有价值。

主要及官方资源包括 Hernán 与 Robins 的免费教材 [Causal Inference: What If](https://www.hsph.harvard.edu/miguel-hernan/causal-inference-book/)、Pearl 对[因果推断基础](https://onlinelibrary.wiley.com/doi/10.1111/j.1467-9531.2010.01228.x)的综述、开放获取教材 [Elements of Causal Inference](https://mitpress.mit.edu/9780262344296/elements-of-causal-inference/)、[Stanford Machine Learning & Causal Inference 课程](https://www.gsb.stanford.edu/faculty-research/labs-initiatives/sil/research/methods/ai-machine-learning/short-course)、[MIT Causal Inference](https://computing.mit.edu/cross-cutting/common-ground-for-computing-education/common-ground-subjects/c08-causal-inference/)、[DoWhy 四步工作流](https://petergtz.github.io/dowhy/main/getting_started/index.html)、最初的 [Double/Debiased Machine Learning 论文](https://academic.oup.com/ectj/article/21/1/C1/5056401)、[meta-learner 论文](https://arxiv.org/abs/1706.03461)，以及关于 [causal forest](https://arxiv.org/abs/1510.04342) 与 [generalized random forest](https://arxiv.org/abs/1610.01271) 的原始研究。
